# 1. Baseline 실험 - TF-IDF Retrieval + klue/bert-base

본 노트북은 **TF-IDF 기반 Retrieval**과 **klue/bert-base MRC 모델**을 사용하여 빠른 실험을 진행합니다.

**실험 설정:**
- Retrieval: TF-IDF (Sparse Retrieval)
- Reader: klue/bert-base
- 데이터: 기존 학습 데이터셋

## 1.1. 환경 설정

In [ ]:
# 가상환경(.venv) 경로 자동 추가
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent
venv_path = project_root / ".venv"

if venv_path.exists():
    python_version = f"{sys.version_info.major}.{sys.version_info.minor}"
    venv_site_packages = venv_path / "lib" / f"python{python_version}" / "site-packages"
    if venv_site_packages.exists():
        if str(venv_site_packages) not in sys.path:
            sys.path.insert(0, str(venv_site_packages))
        print(f"✅ 가상환경(.venv) 경로 추가됨")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print(f"✅ 프로젝트 루트: {project_root}")

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import json
import random
import numpy as np
import torch
from datasets import load_from_disk
from tqdm.auto import tqdm
import evaluate

from transformers import (
    AutoConfig, AutoModelForQuestionAnswering, AutoTokenizer,
    DataCollatorWithPadding, EvalPrediction, TrainingArguments, set_seed
)

from src.training.trainer_qa import QuestionAnsweringTrainer
from src.utils import postprocess_qa_predictions
from src.retrieval.retrieval import SparseRetrieval

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 디바이스: {device}")

## 1.2. 데이터 로드

In [ ]:
data_root = project_root / "data"
train_dataset_path = data_root / "train_dataset"

experiment_dir = Path().resolve() / "experiments" / "baseline_tfidf"
experiment_dir.mkdir(parents=True, exist_ok=True)

train_datasets = load_from_disk(str(train_dataset_path))
print(f"Train: {len(train_datasets['train'])} samples")
print(f"Validation: {len(train_datasets['validation'])} samples")

## 1.3. 모델 및 토크나이저 설정

In [ ]:
MODEL_NAME = "klue/bert-base"
MAX_SEQ_LENGTH = 384
DOC_STRIDE = 128

print(f"모델 로드 중: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model_config = AutoConfig.from_pretrained(MODEL_NAME)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME, config=model_config)
print(f"모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

## 1.4. TF-IDF Retrieval 설정

In [ ]:
print("TF-IDF Retriever 초기화 중...")
retriever = SparseRetrieval(
    tokenize_fn=tokenizer.tokenize,
    data_path=str(data_root),
    context_path="wikipedia_documents.json",
)
retriever.get_sparse_embedding()
print("✅ TF-IDF Retriever 준비 완료")

In [ ]:
# Retrieval 테스트
test_query = "대한민국의 수도는 어디인가?"
scores, passages = retriever.retrieve(test_query, topk=1)
print(f"Query: {test_query}")
print(f"Score: {scores[0]:.4f}")

## 1.5. Retrieval 성능 확인

In [ ]:
TOP_K = 1
print(f"Validation 데이터에 대해 TF-IDF Retrieval 수행 (Top-{TOP_K})...")
retrieved_df = retriever.retrieve(train_datasets['validation'], topk=TOP_K)

if 'original_context' in retrieved_df.columns:
    def check_answer_in_context(row):
        if row['answers'] and row['answers']['text']:
            return row['answers']['text'][0] in row['context']
        return False
    answer_found = retrieved_df.apply(check_answer_in_context, axis=1).sum()
    print(f"Answer Found: {answer_found}/{len(retrieved_df)} ({answer_found/len(retrieved_df)*100:.2f}%)")

## 1.6. 데이터 전처리

In [ ]:
def prepare_train_features(examples):
    tokenized = tokenizer(
        examples['question'], examples['context'],
        truncation="only_second", max_length=MAX_SEQ_LENGTH,
        stride=DOC_STRIDE, return_overflowing_tokens=True,
        return_offsets_mapping=True, padding="max_length",
    )
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")
    tokenized["start_positions"] = []
    tokenized["end_positions"] = []
    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized.sequence_ids(i)
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]
        if len(answers["answer_start"]) == 0:
            tokenized["start_positions"].append(cls_index)
            tokenized["end_positions"].append(cls_index)
        else:
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])
            token_start_index = 0
            while sequence_ids[token_start_index] != 1:
                token_start_index += 1
            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != 1:
                token_end_index -= 1
            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                tokenized["start_positions"].append(cls_index)
                tokenized["end_positions"].append(cls_index)
            else:
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                tokenized["start_positions"].append(token_start_index - 1)
                while offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                tokenized["end_positions"].append(token_end_index + 1)
    return tokenized

def prepare_validation_features(examples):
    tokenized = tokenizer(
        examples['question'], examples['context'],
        truncation="only_second", max_length=MAX_SEQ_LENGTH,
        stride=DOC_STRIDE, return_overflowing_tokens=True,
        return_offsets_mapping=True, padding="max_length",
    )
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    tokenized["example_id"] = []
    for i in range(len(tokenized["input_ids"])):
        sequence_ids = tokenized.sequence_ids(i)
        sample_index = sample_mapping[i]
        tokenized["example_id"].append(examples["id"][sample_index])
        tokenized["offset_mapping"][i] = [
            (o if sequence_ids[k] == 1 else None)
            for k, o in enumerate(tokenized["offset_mapping"][i])
        ]
    return tokenized

In [ ]:
print("학습 데이터 전처리 중...")
train_dataset = train_datasets['train'].map(
    prepare_train_features, batched=True,
    remove_columns=train_datasets['train'].column_names
)
print("검증 데이터 전처리 중...")
validation_dataset = train_datasets['validation'].map(
    prepare_validation_features, batched=True,
    remove_columns=train_datasets['validation'].column_names
)
print(f"전처리된 학습 데이터: {len(train_dataset)}")
print(f"전처리된 검증 데이터: {len(validation_dataset)}")

## 1.7. 학습 설정 및 실행

In [ ]:
training_args = TrainingArguments(
    output_dir=str(experiment_dir),
    do_train=True, do_eval=True,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01, warmup_ratio=0.1,
    logging_steps=100,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1", greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none", seed=SEED,
)

data_collator = DataCollatorWithPadding(
    tokenizer, pad_to_multiple_of=8 if training_args.fp16 else None
)

print(f"Model: {MODEL_NAME}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch Size: {training_args.per_device_train_batch_size}")

In [ ]:
def post_processing_function(examples, features, predictions, stage="eval"):
    predictions = postprocess_qa_predictions(
        examples=examples, features=features, predictions=predictions,
        max_answer_length=30, output_dir=str(experiment_dir),
    )
    formatted_predictions = [{"id": k, "prediction_text": v} for k, v in predictions.items()]
    if stage == "predict":
        return formatted_predictions
    references = [{"id": ex["id"], "answers": ex["answers"]} for ex in train_datasets['validation']]
    return EvalPrediction(predictions=formatted_predictions, label_ids=references)

metric = evaluate.load("squad")
def compute_metrics(p: EvalPrediction):
    result = metric.compute(predictions=p.predictions, references=p.label_ids)
    return {f"eval_{k}": v for k, v in result.items()}

In [ ]:
trainer = QuestionAnsweringTrainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=validation_dataset,
    eval_examples=train_datasets['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    post_process_function=post_processing_function,
    compute_metrics=compute_metrics,
)
print("✅ Trainer 초기화 완료")

In [ ]:
print("="*50)
print("Baseline 학습 시작 (TF-IDF + klue/bert-base)")
print("="*50)

train_result = trainer.train()
trainer.save_model()
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
print(f"\nTrain Loss: {metrics.get('train_loss', 'N/A'):.4f}")

## 1.8. 평가

In [ ]:
print("="*50)
print("평가 시작")
print("="*50)

eval_metrics = trainer.evaluate()
trainer.log_metrics("eval", eval_metrics)
trainer.save_metrics("eval", eval_metrics)

print(f"\n=== 결과 ===")
print(f"EM: {eval_metrics.get('eval_exact_match', 'N/A'):.2f}")
print(f"F1: {eval_metrics.get('eval_f1', 'N/A'):.2f}")

In [ ]:
results_summary = {
    "experiment": "baseline_tfidf",
    "retrieval": "TF-IDF",
    "model": MODEL_NAME,
    "train_samples": len(train_datasets['train']),
    "eval_samples": len(train_datasets['validation']),
    "epochs": training_args.num_train_epochs,
    "eval_exact_match": eval_metrics.get('eval_exact_match'),
    "eval_f1": eval_metrics.get('eval_f1'),
}

results_path = experiment_dir / "results_summary.json"
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(results_summary, f, ensure_ascii=False, indent=2)
print(f"결과 저장: {results_path}")